# A relation container is never its entity

An order holds its customer through an `AssociationOne`. The container passes ordinary names to the loaded customer, so `order.customer.name` reads the name, but it answers Python's and Pydantic's own questions itself: a Python dump keeps it, a JSON dump writes its form, a copy and a pickle give a container back, loaded or not.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [ ]:
!pip install -q aoa-action-machine

In [2]:
import copy
import pickle
from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import AssociationMany, AssociationOne, BaseCoreEntity, NoInverse
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entities import entities
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.resources.base_storage import BaseStorage

## Declare the model

In [3]:
class ShopDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Customer", domain=ShopDomain)
class CustomerCoreEntity(BaseCoreEntity):
    """Describe a customer."""

    id: str = Field(description="Customer identifier")
    name: str = Field(description="Customer name")


@entity(description="Customer order", domain=ShopDomain)
class OrderCoreEntity(BaseCoreEntity):
    """Describe an order that points at its customer."""

    id: str = Field(description="Order identifier")
    customer: Annotated[AssociationOne[CustomerCoreEntity], NoInverse()] = Field(description="Customer")


@meta(description="Stores the shop entities")
@entities(
    CustomerCoreEntity,
    OrderCoreEntity,
)
class ShopStorage(BaseStorage):
    """Hold the shop entities."""

## Ordinary names and protocol names

In [4]:
customer = CustomerCoreEntity(id="cust-1", name="Alice")
order = OrderCoreEntity(id="ord-1", customer=AssociationOne[CustomerCoreEntity](id="cust-1", entity=customer))

print("1) Names")
print("   order.customer.name:", order.customer.name)
try:
    _ = order.customer.__pydantic_serializer__
except AttributeError:
    print("   order.customer.__pydantic_serializer__ -> AttributeError")
else:
    raise AssertionError("A protocol name must not reach the customer")

1) Names
   order.customer.name: Alice
   order.customer.__pydantic_serializer__ -> AttributeError


## Dumps

In [5]:
print("2) Dumps")
print("   Python dump keeps the container:", order.model_dump()["customer"])
print("   JSON dump writes its form:", order.model_dump(mode="json")["customer"])

2) Dumps
   Python dump keeps the container: AssociationOne(id='cust-1', loaded)
   JSON dump writes its form: {'id': 'cust-1', 'entity': {'id': 'cust-1', 'name': 'Alice'}}


## Copies

In [6]:
print("3) Copies")
deep = copy.deepcopy(order.customer)
print("   deepcopy:", deep, "| customer copied:", deep.entity == customer and deep.entity is not customer)
print("   model_copy(deep=True).customer:", order.model_copy(deep=True).customer)
print("   copy.copy is equal:", copy.copy(order.customer) == order.customer)

3) Copies
   deepcopy: AssociationOne(id='cust-1', loaded) | customer copied: True
   model_copy(deep=True).customer: AssociationOne(id='cust-1', loaded)
   copy.copy is equal: True


## Pickle

In [7]:
print("4) Pickle")
unloaded = AssociationOne[CustomerCoreEntity](id="cust-9")
back = pickle.loads(pickle.dumps(unloaded))
print("   unloaded one:", back, "| equal:", back == unloaded)
many = AssociationMany[CustomerCoreEntity](ids=("cust-1", "cust-2"))
print("   unloaded many:", pickle.loads(pickle.dumps(many)))
try:
    back.extra = 1
except AttributeError as exc:
    print("   still frozen:", exc)

4) Pickle
   unloaded one: AssociationOne(id='cust-9', id_only) | equal: True
   unloaded many: AssociationMany(count=2, entities_loaded=no, n_entities=0)
   still frozen: AssociationOne is frozen; assigning to 'extra' is not allowed.


A name that starts and ends with two underscores belongs to Python or Pydantic and is never passed on, so the container is dumped, copied and pickled as a container. A JSON dump writes the relation's form, `{"id", "entity"}` when it is loaded.